# Modelling

Date: 12/03/2026

This notebook is used for modelling and processing data which has been anaylsed in the EDA notebook. We will preprocess the data, select features and test models with hyperparamter tuning. 

In [8]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split

In [5]:
df_clean = pd.read_csv('../data/processed/df_clean.csv') # import our cleaned data

In [11]:
X = df_clean.drop(columns=['DEFAULT']) # remove target from out features
y = df_clean['DEFAULT'] 


print('X shape: ', X.shape)
print('y shape: ', y.shape)

X shape:  (29601, 23)
y shape:  (29601,)


We split our data into training and testing sample with a test_size of 0.2. This leaves around 5,921 instances for testing enough to meaningfully evaluate recall and other metrics. We use stratify=y here to ensure the training sample is representative of the total data set class balance.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13, stratify=y)

In [17]:
y_train.describe()

count    23680.000000
mean         0.223142
std          0.416362
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: DEFAULT, dtype: float64

In [19]:
X_train.columns

Index(['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2',
       'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'],
      dtype='object')

In [ ]:
y_train.describe()

count    23680.000000
mean         0.223142
std          0.416362
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: DEFAULT, dtype: float64

In [20]:
num_cols = ['LIMIT_BAL', 'AGE','BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
             'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 
            'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6' ]

cat_cols = ['SEX', 'MARRIAGE']

ordinal_cols = ['EDUCATION',  'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']



In [25]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer

Creating the sub-pipelines. The numerical pipe with for transformed using the PowerTransformer (with method='yeo-johnson'), which handles negative values unlike a log transformation. The data is then scaled. The categories are one hot encoded and the ordinal categories are kept as they are. 

In [28]:
# numerical pipeline: PowerTransformer -> StandardScaler
num_pipe = Pipeline(steps=[
    ('power_trans', PowerTransformer(method='yeo-johnson')),
    ('scale', StandardScaler()),
])

# categorical pipeline: OneHotEncoder
cat_pipe = Pipeline(steps=[
    ('one_hot', OneHotEncoder())
])

In [30]:
preprocessor = ColumnTransformer(transformers=[
    ('num_pipe', num_pipe, num_cols),
    ('cat_pipe', cat_pipe, cat_cols),
    ('ordinal', 'passthrough', ordinal_cols) # ordinal columns are just passed through 
])

In [31]:
from sklearn.dummy import DummyClassifier

dummy_clf = DummyClassifier(strategy='most_frequent')

dummy_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('dummy', dummy_clf)
])

In [33]:
dummy_fit = dummy_pipeline.fit(X_train, y_train)

dummy_pred = dummy_fit.predict(X_test)

In [35]:
from sklearn.metrics import classification_report

dummy_report = classification_report(y_test, dummy_pred)

/Users/shakurahmad/PythonProjects/credit-risk-project/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/shakurahmad/PythonProjects/credit-risk-project/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/shakurahmad/PythonProjects/credit-risk-project/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to c

In [36]:
print(dummy_report)

              precision    recall  f1-score   support

           0       0.78      1.00      0.87      4600
           1       0.00      0.00      0.00      1321

    accuracy                           0.78      5921
   macro avg       0.39      0.50      0.44      5921
weighted avg       0.60      0.78      0.68      5921



We have used a dummy classifier to generate a baseline result for precision and recall. Precision, recall and f1 are all 0 for defaulters. The model never predicts a default. 